# Weather Prediction Pipeline — Data Ingestion (Databricks)

Loads the two data sources from the project proposal:

1. **NOAA CDO (Climate Data Online)** — historical daily station data, used for training/test data. Requires a free API token.
2. **api.weather.gov (NWS)** — live/current observations and forecasts, no key required. Used later for the production/monitoring and drift-simulation stages.

> This notebook only covers **Stage 1: Data Ingestion & Baselines**. Preprocessing, MLflow tracking, etc. live in separate notebooks per the project outline.

> **This is the one-time historical backfill.** It rebuilds the bronze tables from scratch, so it is run by hand, not scheduled. The nightly incremental refresh is `data_pipelines/10_ingest_nightly.ipynb`, which merges rather than rebuilds.

## Prerequisites (one-time, before running this notebook)

1. Get a free NOAA CDO token: https://www.ncdc.noaa.gov/cdo-web/token
2. Store it in a Databricks secret scope (never hardcode it in the notebook):
   ```bash
   databricks secrets create-scope mlo
   databricks secrets put-secret mlo WEATHER_API_KEY
   ```
3. Attach this notebook to a cluster/SQL warehouse with Unity Catalog access if you want the Delta writes in section 4 to succeed, and update the `catalog` widget if you're not using `mlo`.

All other run-time parameters (location, date range, coordinates, catalog/schema) are exposed as notebook widgets in section 0.

## 0. Setup

In [0]:
import time
import requests
import pandas as pd
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)

# Notebook widgets: configurable per-run (or via a Databricks Job/Workflow task's parameters)
dbutils.widgets.text("location_id", "FIPS:17031", "NOAA locationid (station discovery only)")
dbutils.widgets.text("station_ids", "", "Stations to backfill, comma separated (blank = module default)")
dbutils.widgets.text("datatypes", "", "NOAA datatypes, comma separated (blank = module default)")
dbutils.widgets.text("start_date", "2020-01-01", "Start date")
dbutils.widgets.text("end_date", "2024-12-31", "End date")
dbutils.widgets.text("lat", "41.85", "NWS latitude")
dbutils.widgets.text("lon", "-87.65", "NWS longitude")
dbutils.widgets.text("nws_contact_email", "klkendall@uchicago.edu", "Contact email for NWS User-Agent")
dbutils.widgets.text("catalog", "mlo", "Unity Catalog catalog")
dbutils.widgets.text("schema", "weather_mlops", "Schema for bronze tables")

## 1. NOAA CDO — Historical Data

Get a free token here: https://www.ncdc.noaa.gov/cdo-web/token

**Do not hardcode your token in the notebook.** It's read from the Databricks secret scope `mlo`, key `WEATHER_API_KEY` — see the Prerequisites section above for how to create it.

In [0]:
import os
import sys

# Files in Repos puts this notebook's own directory — the repo root — on sys.path,
# which makes the `data_pipelines` package importable. The fetch/pagination/retry
# helpers live there so this backfill and the nightly incremental job pull identically.
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from data_pipelines.noaa_client import (
    DEFAULT_DATATYPES,
    DEFAULT_STATIONS,
    get_ghcnd_daily_multi,
    get_stations,
    make_headers,
)

NOAA_TOKEN = dbutils.secrets.get(scope="mlo", key="WEATHER_API_KEY")
HEADERS = make_headers(NOAA_TOKEN)


def _csv_widget(name, fallback):
    raw = dbutils.widgets.get(name).strip()
    return tuple(v.strip() for v in raw.split(",") if v.strip()) if raw else tuple(fallback)


# Blank widgets fall back to the module. The station list and the column contract live in
# version control rather than in job parameters — the nightly MERGE depends on bronze
# matching both, so a silent parameter change here would break it at 3am, not now.
STATION_IDS = _csv_widget("station_ids", DEFAULT_STATIONS)
DATATYPES = _csv_widget("datatypes", DEFAULT_DATATYPES)

print(f"{len(STATION_IDS)} stations, {len(DATATYPES)} datatypes")
print("datatypes:", ", ".join(DATATYPES))

### 1.1 Station discovery (optional)

Lists GHCND stations for the `location_id` widget. This is a convenience lookup only — the backfill pulls the pinned list in `noaa_client.DEFAULT_STATIONS`, not whatever this cell returns.

For actually *choosing* the station list, use `data_pipelines/00_explore_noaa_catalog.ipynb`: it sweeps multiple metros by bounding box, filters to first-order airport stations, and measures per-datatype density so the choice is made on real coverage rather than on NOAA's `datacoverage` field.

In [0]:
# Chicago, IL -> FIPS location id 'FIPS:17031' (Cook County) by default. Override via the location_id widget.
LOCATION_ID = dbutils.widgets.get("location_id")

print(LOCATION_ID)

stations_df = get_stations(LOCATION_ID, headers=HEADERS)
stations_df[["id", "name", "mindate", "maxdate", "datacoverage"]].head(10) if not stations_df.empty else stations_df

### 1.2 Pull historical daily data (GHCND)

`datasetid=GHCND` (Global Historical Climatology Network - Daily). The frozen column contract is `noaa_client.DEFAULT_DATATYPES`: eight daily measurements (`AWND`, `PRCP`, `TMAX`, `TMIN`, `WDF2`, `WDF5`, `WSF2`, `WSF5`), plus `SNWD`, plus the weather-type flags `WT01`/`WT02`/`WT03`.

The API caps each request to a **1-year date range and 1000 records**. The pagination, the 429 retry/backoff, the long→wide reshape, and the multi-station loop all live in `data_pipelines/noaa_client.py`, shared with the nightly incremental job so both write identically-shaped rows into the same bronze table.

In [0]:
# The backfill pulls the pinned station list rather than choosing a station dynamically:
# bronze has to contain exactly what the nightly job expects to merge into. Choosing which
# stations belong on that list is the job of data_pipelines/00_explore_noaa_catalog.ipynb.
START_DATE, END_DATE = dbutils.widgets.get("start_date"), dbutils.widgets.get("end_date")

print(f"backfilling {START_DATE}..{END_DATE} — expect a few minutes, NOAA is rate-limited\n")

noaa_daily, failures = get_ghcnd_daily_multi(
    STATION_IDS,
    START_DATE,
    END_DATE,
    headers=HEADERS,
    datatypes=DATATYPES,
    progress=lambda station_id, n: print(f"  {station_id:<20} {n:>5} days"),
)

# skip_errors stays False here, unlike the nightly job. A partial history written silently is
# worse than a failed run, because everything downstream would train on it without knowing.
assert not failures, failures

print(f"\n{len(noaa_daily)} station-days total")
noaa_daily.head()

In [0]:
# to_wide() already ran inside get_ghcnd_daily_multi, so noaa_daily is in the frozen shape:
# long in the station dimension, one column per datatype, every datatype guaranteed present.
#
# Per-station day counts are the thing to eyeball before this reaches bronze — a station that
# quietly returned far fewer days than the rest is the failure this catches.
per_station = (noaa_daily
               .groupby("station")
               .agg(days=("date", "size"), first_date=("date", "min"), last_date=("date", "max"))
               .sort_values("days"))

print(per_station.to_string())

## 2. api.weather.gov — Live Data

No API key required — only a descriptive `User-Agent` header (NWS policy). This will later feed the production inference + drift-simulation stages, but we pull a sample here to confirm access and shape.

In [0]:
NWS_HEADERS = {
    "User-Agent": f"(mlops-weather-project, contact: {dbutils.widgets.get('nws_contact_email')})",
    "Accept": "application/geo+json",
}

# Same Chicago-area point used above, as lat/lon. Override via the lat / lon widgets.
LAT, LON = float(dbutils.widgets.get("lat")), float(dbutils.widgets.get("lon"))

def get_nws_point_metadata(lat, lon):
    resp = requests.get(f"https://api.weather.gov/points/{lat},{lon}", headers=NWS_HEADERS)
    resp.raise_for_status()
    return resp.json()

point_meta = get_nws_point_metadata(LAT, LON)
forecast_url = point_meta["properties"]["forecast"]
forecast_hourly_url = point_meta["properties"]["forecastHourly"]
stations_url = point_meta["properties"]["observationStations"]
forecast_url, forecast_hourly_url, stations_url


In [0]:
def get_nws_forecast(forecast_url):
    resp = requests.get(forecast_url, headers=NWS_HEADERS)
    resp.raise_for_status()
    periods = resp.json()["properties"]["periods"]
    # json_normalize flattens nested fields (dewpoint, relativeHumidity, probabilityOfPrecipitation
    # come back as {"unitCode": ..., "value": ...} objects) into dot-separated columns so the
    # result is Spark/Delta-friendly downstream.
    return pd.json_normalize(periods)

nws_forecast = get_nws_forecast(forecast_url)
nws_forecast[["name", "startTime", "temperature", "temperatureUnit", "windSpeed", "shortForecast"]].head(10)


In [0]:
def get_nws_latest_observation(stations_url):
    stations_resp = requests.get(stations_url, headers=NWS_HEADERS)
    stations_resp.raise_for_status()
    nearest_station_id = stations_resp.json()["features"][0]["properties"]["stationIdentifier"]

    obs_resp = requests.get(
        f"https://api.weather.gov/stations/{nearest_station_id}/observations/latest",
        headers=NWS_HEADERS,
    )
    obs_resp.raise_for_status()
    return nearest_station_id, obs_resp.json()["properties"]

nearest_station_id, latest_obs = get_nws_latest_observation(stations_url)
print("Nearest station:", nearest_station_id)
pd.json_normalize(latest_obs)[["timestamp", "temperature.value", "windSpeed.value", "relativeHumidity.value", "textDescription"]]


## 3. Sanity checks

Quick shape/null checks before moving to preprocessing.

In [0]:
print("NOAA historical daily shape:", noaa_daily.shape)
print()

# High null rates on WT01/WT02/WT03 are expected and correct, not a data problem: NOAA only
# writes a weather-type flag on days the phenomenon occurred. Null means "did not happen",
# so downstream these get filled with 0 — never dropped. SNWD is similar but seasonal.
print(noaa_daily.isna().mean().round(3))
print()
print("NWS forecast periods shape:", nws_forecast.shape)

In [0]:
noaa_daily.groupby('station')[list(DATATYPES)].apply(lambda g: g.isna().mean().round(2))

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS mlo;

## 4. Save raw pulls as Delta tables

Persist raw pulls as managed Delta tables (bronze layer) before any cleaning/versioning happens downstream (e.g. in the feature-store step). Target catalog/schema are controlled by the `catalog` / `schema` widgets.

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

print(CATALOG)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

noaa_table = f"{CATALOG}.{SCHEMA}.noaa_historical_daily"
nws_table = f"{CATALOG}.{SCHEMA}.nws_forecast_snapshot"


spark.sql(f"DROP TABLE IF EXISTS {noaa_table}")

noaa_spark_df = spark.createDataFrame(noaa_daily)
noaa_spark_df.write.mode("overwrite").saveAsTable(noaa_table)

nws_spark_df = spark.createDataFrame(nws_forecast)
nws_spark_df.write.mode("overwrite").saveAsTable(nws_table)

print(f"Saved NOAA historical daily -> {noaa_table} ({noaa_spark_df.count()} rows)")
print(f"Saved NWS forecast snapshot -> {nws_table} ({nws_spark_df.count()} rows)")


---
**Next steps** (per the project outline):
- Recurring ingestion is `data_pipelines/10_ingest_nightly.ipynb`, **not** this notebook — this one rebuilds bronze from scratch and is run by hand
- Bronze now holds multiple stations. Anything reading it that expects a single station must filter explicitly — see the `PRIMARY_STATION` filter added to `02_feature_store.ipynb`
- Define target variable + train/test split (Stage 1 continued), reading from the `noaa_historical_daily` bronze table
- Track dataset version alongside MLflow experiment runs

In [0]:
spark.table(noaa_table).printSchema()

In [0]:
%sql
SELECT * FROM mlo.weather_mlops.noaa_historical_daily
LIMIT 5;

## 5. Test

Final check on full pull.

In [0]:
%sql
SELECT station, COUNT(*) AS rows, MIN(date) AS min_date, MAX(date) AS max_date
FROM mlo.weather_mlops.noaa_historical_daily GROUP BY station ORDER BY station;